# LLM-Based ICF Linking of Patient Goal Concepts
## Results Overview

Comparing three LLM linking conditions on 470 held-out test concepts across four German rehabilitation cohorts:

| Option | Method |
|--------|--------|
| **A — Zero-shot** | Concept + full ICF code list (1,454 codes) |
| **B — Few-shot** | Concept + 8 fixed expert-annotated examples + full ICF code list |
| **C — RAG** | Concept + top-30 retrieved candidate codes + top-6 retrieved similar examples |

**Three evaluation metrics:**
- **Exact match** — predicted code set == gold code set
- **One-level-up** — 4-digit prefix matches (e.g. `d4500` accepted for gold `d450`)
- **Component match** — correct ICF component (b / s / d / e)

**Section 10:** LLM comparison — 8 models evaluated under RAG with the same retrieval index  
**Section 11:** Embedding model comparison — qwen3-embedding:8b vs bge-m3  

---
*Data: Akut (Müller 2011), Geriat (Kus 2011), PostAkut (Lohmann 2011), Vertigo (Müller 2012)*  
*Reference codebook: BfArM ICF 2005 (1,454 codes)*  
*Embedding: qwen3-embedding:8b (4096-dim) | Primary LLM: qwen3.6:35b*

## Setup

In [ ]:
from __future__ import annotations

import json
import sys
import os
from pathlib import Path
from collections import defaultdict
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── paths ──────────────────────────────────────────────────────────────────
REPO = Path("__file__").resolve().parent.parent
sys.path.insert(0, str(REPO / "scripts"))

OUTPUT  = REPO / "output"
DATASET = REPO / "dataset"

from utils.data_utils import load_splits, load_icf_reference
from utils.eval_utils  import score_prediction, NON_STANDARD

# ── plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = {"A_ZeroShot": "#4C72B0", "B_FewShot": "#DD8452", "C_RAG": "#55A868"}
COHORT_ORDER = ["Akut", "Geriat", "PostAkut", "Vertigo", "ALL"]

# ── helpers ────────────────────────────────────────────────────────────────
def load_results(name: str) -> Optional[list]:
    path = OUTPUT / f"results_{name}.jsonl"
    if not path.exists():
        print(f"  ⚠ {path.name} not found — run the corresponding script first.")
        return None
    rows = []
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            scores = score_prediction(rec["predicted_codes"], rec["gold_codes"])
            rec.update(scores)
            rows.append(rec)
    return rows

def summary_df(results: list) -> pd.DataFrame:
    df = pd.DataFrame(results)
    rows = []
    for cohort in COHORT_ORDER:
        grp = df if cohort == "ALL" else df[df["cohort"] == cohort]
        if len(grp) == 0:
            continue
        rows.append({
            "Cohort": cohort,
            "N": len(grp),
            "Exact %": grp["exact"].mean() * 100,
            "One-level-up %": grp["one_level_up"].mean() * 100,
            "Component %": grp["component"].mean() * 100,
        })
    return pd.DataFrame(rows).set_index("Cohort")

print("Setup complete.")
print(f"Repo root : {REPO}")
print(f"Output dir: {OUTPUT}")

---
## 1. Dataset Overview

In [ ]:
train_df, test_df = load_splits()
icf_df = load_icf_reference()

# ── cohort split table ──────────────────────────────────────────────────────
split_rows = []
for cohort in ["Akut", "Geriat", "PostAkut", "Vertigo"]:
    tr = train_df[train_df["cohort"] == cohort]
    te = test_df[test_df["cohort"] == cohort]
    split_rows.append({
        "Cohort": cohort,
        "Total": len(tr) + len(te),
        "Train (80%)": len(tr),
        "Test (20%)": len(te),
        "Unique codes (train)": train_df[train_df["cohort"] == cohort]["primary_code"].nunique(),
    })

split_df = pd.DataFrame(split_rows).set_index("Cohort")
split_df.loc["TOTAL"] = split_df.sum()
split_df.loc["TOTAL", "Unique codes (train)"] = train_df["primary_code"].nunique()

print("=== Cohort Split ===")
display(split_df.style.set_caption("Train / Test split per cohort"))

In [ ]:
# ── ICF component distribution in test set ─────────────────────────────────
all_codes = [c for codes in test_df["gold_codes"] for c in codes]
std_codes  = [c for c in all_codes if c not in NON_STANDARD and c[0] in "bsde"]
nonstd     = [c for c in all_codes if c in NON_STANDARD]

comp_counts = pd.Series([c[0] for c in std_codes]).value_counts().reindex(["b","s","d","e"], fill_value=0)
comp_labels = {"b": "Body Functions (b)", "s": "Body Structures (s)",
               "d": "Activities & Participation (d)", "e": "Environmental (e)"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Component pie
wedge_data = list(comp_counts.values) + [len(nonstd)]
wedge_labels = [comp_labels[k] for k in comp_counts.index] + ["Non-standard (nd/nc/pf/hc)"]
axes[0].pie(wedge_data, labels=wedge_labels, autopct="%1.1f%%",
            colors=sns.color_palette("Set2", len(wedge_data)), startangle=140)
axes[0].set_title("ICF Component Distribution\n(test set gold codes)")

# Top-20 most frequent codes
code_freq = pd.Series(all_codes).value_counts().head(20)
icf_lookup = dict(zip(icf_df["code"], icf_df["title_de"]))
code_freq.index = [f"{c} — {icf_lookup.get(c, c)[:28]}" for c in code_freq.index]
code_freq.plot(kind="barh", ax=axes[1], color="#4C72B0")
axes[1].invert_yaxis()
axes[1].set_xlabel("Frequency")
axes[1].set_title("Top 20 ICF Codes in Test Set")

plt.tight_layout()
plt.savefig(OUTPUT / "fig_data_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Total test codes: {len(all_codes)} | Standard: {len(std_codes)} | Non-standard: {len(nonstd)}")

---
## 2. Option A — Zero-shot Results

In [ ]:
res_a = load_results("option_a")

if res_a:
    df_a = summary_df(res_a)
    display(
        df_a.style
          .format("{:.1f}", subset=["Exact %", "One-level-up %", "Component %"])
          .background_gradient(cmap="YlGn", subset=["Exact %", "One-level-up %", "Component %"])
          .set_caption("Option A — Zero-shot: accuracy per cohort (%)")
    )

---
## 3. Option B — Few-shot Results

In [ ]:
# Show the 8 fixed examples used in Option B
ex_path = OUTPUT / "option_b_fixed_examples.json"
if ex_path.exists():
    examples = json.loads(ex_path.read_text())
    ex_df = pd.DataFrame([
        {"#": i+1, "Cohort": ex["cohort"],
         "Konzept": ex["konzept"],
         "Gold ICF codes": ", ".join(ex["gold_codes"])}
        for i, ex in enumerate(examples)
    ]).set_index("#")
    display(ex_df.style.set_caption("Fixed few-shot examples used for all Option B predictions"))
else:
    print("Fixed examples file not found — run 04_fewshot.py first.")

In [ ]:
res_b = load_results("option_b")

if res_b:
    df_b = summary_df(res_b)
    display(
        df_b.style
          .format("{:.1f}", subset=["Exact %", "One-level-up %", "Component %"])
          .background_gradient(cmap="YlGn", subset=["Exact %", "One-level-up %", "Component %"])
          .set_caption("Option B — Few-shot (N=8): accuracy per cohort (%)")
    )

---
## 4. Option C — RAG Results

In [ ]:
res_c = load_results("option_c")

if res_c:
    df_c = summary_df(res_c)
    display(
        df_c.style
          .format("{:.1f}", subset=["Exact %", "One-level-up %", "Component %"])
          .background_gradient(cmap="YlGn", subset=["Exact %", "One-level-up %", "Component %"])
          .set_caption("Option C — RAG (top-30 codes + top-6 examples): accuracy per cohort (%)")
    )

---
## 5. Comparison: A vs B vs C

In [ ]:
options = {"A_ZeroShot": res_a, "B_FewShot": res_b, "C_RAG": res_c}
available = {k: v for k, v in options.items() if v is not None}

if available:
    # Build wide comparison table
    frames = []
    for label, res in available.items():
        s = summary_df(res)[["Exact %", "One-level-up %", "Component %"]]
        s.columns = pd.MultiIndex.from_tuples([(label, c) for c in s.columns])
        frames.append(s)

    comparison = pd.concat(frames, axis=1)
    comparison.to_csv(OUTPUT / "comparison_table.csv")

    display(
        comparison.style
          .format("{:.1f}")
          .background_gradient(cmap="YlGn", axis=None)
          .set_caption("Accuracy (%) — A vs B vs C per cohort and metric")
    )
    print("Table saved → output/comparison_table.csv")

In [ ]:
if available:
    metrics = ["Exact %", "One-level-up %", "Component %"]
    cohorts = [c for c in COHORT_ORDER if c in comparison.index]
    n_cohorts = len(cohorts)
    n_opts = len(available)
    bar_w = 0.22

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

    for ax, metric in zip(axes, metrics):
        x = np.arange(n_cohorts)
        for i, (label, _) in enumerate(available.items()):
            vals = [comparison.loc[c, (label, metric)] if c in comparison.index else 0
                    for c in cohorts]
            offset = (i - (n_opts - 1) / 2) * bar_w
            bars = ax.bar(x + offset, vals, bar_w,
                          label=label.replace("_", " "),
                          color=PALETTE.get(label, f"C{i}"),
                          edgecolor="white", linewidth=0.5)
            for bar, v in zip(bars, vals):
                if v > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.8,
                            f"{v:.0f}", ha="center", va="bottom",
                            fontsize=7.5, color="#333")

        ax.set_title(metric, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(cohorts, rotation=15)
        ax.set_ylabel("Accuracy (%)")
        ax.set_ylim(0, 108)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g%%"))
        ax.legend(fontsize=8)

    fig.suptitle("ICF Linking Accuracy: Zero-shot vs Few-shot vs RAG",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT / "fig_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved → output/fig_comparison.png")

---
## 6. Per-ICF-Component Breakdown (best option)

In [ ]:
# Use best available option for component breakdown
best_label, best_res = list(available.items())[-1] if available else (None, None)

if best_res:
    df_best = pd.DataFrame(best_res)

    def get_component(codes):
        for c in codes:
            if c not in NON_STANDARD and c and c[0] in "bsde":
                return c[0].upper()
        return "Non-std"

    df_best["icf_component"] = df_best["gold_codes"].apply(get_component)
    comp_labels_map = {"B": "Body Functions (b)", "S": "Body Structures (s)",
                       "D": "Activities & Participation (d)",
                       "E": "Environmental (e)", "Non-std": "Non-standard"}
    df_best["component_label"] = df_best["icf_component"].map(comp_labels_map)

    comp_rows = []
    for comp, grp in df_best.groupby("component_label"):
        if len(grp) < 3:
            continue
        comp_rows.append({
            "Component": comp,
            "N": len(grp),
            "Exact %": grp["exact"].mean() * 100,
            "One-level-up %": grp["one_level_up"].mean() * 100,
            "Component %": grp["component"].mean() * 100,
        })
    comp_df = pd.DataFrame(comp_rows).set_index("Component")

    display(
        comp_df.style
          .format("{:.1f}", subset=["Exact %", "One-level-up %", "Component %"])
          .background_gradient(cmap="YlGn", subset=["Exact %", "One-level-up %", "Component %"])
          .set_caption(f"Accuracy by ICF component — {best_label}")
    )

---
## 7. Error Analysis — Sample Predictions

In [ ]:
if best_res:
    df_best = pd.DataFrame(best_res)

    print(f"=== Correct Predictions (exact match) — {best_label} ===")
    correct = df_best[df_best["exact"] == True].sample(
        min(8, df_best["exact"].sum()), random_state=42
    )[["cohort", "konzept", "gold_codes", "predicted_codes"]]
    correct.columns = ["Cohort", "Konzept", "Gold", "Predicted"]
    display(correct.reset_index(drop=True).style.set_caption(
        f"Sample correct predictions ({best_label})"
    ))

In [ ]:
if best_res:
    print(f"=== Incorrect Predictions (exact mismatch) — {best_label} ===")
    wrong = df_best[df_best["exact"] == False].sample(
        min(8, (~df_best["exact"]).sum()), random_state=42
    )[["cohort", "konzept", "gold_codes", "predicted_codes", "one_level_up"]]
    wrong.columns = ["Cohort", "Konzept", "Gold", "Predicted", "1-level-up ✓"]
    display(wrong.reset_index(drop=True).style
        .map(lambda v: "background-color: #d4edda" if v is True else "",
             subset=["1-level-up ✓"])
        .set_caption(f"Sample incorrect predictions ({best_label})"))

---
## 8. Non-standard Code Analysis

In [ ]:
if best_res:
    df_best = pd.DataFrame(best_res)
    ns_mask = df_best["gold_codes"].apply(
        lambda codes: any(c in NON_STANDARD for c in codes)
    )
    ns_df  = df_best[ns_mask]
    std_df = df_best[~ns_mask]

    rows = []
    for label2, subset in [("Standard ICF codes", std_df), ("Non-standard codes (nd/nc/pf/hc)", ns_df)]:
        if len(subset) == 0:
            continue
        rows.append({
            "Group": label2,
            "N": len(subset),
            "Exact %": subset["exact"].mean() * 100,
            "One-level-up %": subset["one_level_up"].mean() * 100,
            "Component %": subset["component"].mean() * 100,
        })

    display(
        pd.DataFrame(rows).set_index("Group").style
          .format("{:.1f}", subset=["Exact %", "One-level-up %", "Component %"])
          .background_gradient(cmap="YlGn", subset=["Exact %", "One-level-up %", "Component %"])
          .set_caption(f"Standard vs non-standard code accuracy — {best_label}")
    )

---
## 9. Cross-cohort Consistency Audit (Option D)

In [ ]:
audit_path = OUTPUT / "consistency_audit.jsonl"

if not audit_path.exists():
    print("Consistency audit not run yet — run 07_consistency_audit.py first.")
else:
    audit = [json.loads(l) for l in open(audit_path)]
    print(f"Total inconsistent groups found: {len(audit)}")

    # Summary by severity
    severity_order = ["component", "mixed", "chapter", "granularity", "qualifier"]
    sev_counts = pd.Series([r["disagreement_level"] for r in audit]).value_counts()
    sev_counts = sev_counts.reindex(severity_order).dropna().astype(int)
    cross_count = sum(1 for r in audit if r["cross_cohort"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Severity bar
    colors = ["#d62728", "#ff7f0e", "#e377c2", "#2ca02c", "#17becf"]
    sev_counts.plot(kind="bar", ax=axes[0],
                    color=colors[:len(sev_counts)], edgecolor="white")
    axes[0].set_title("Inconsistencies by Severity Level")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Number of groups")
    axes[0].tick_params(axis="x", rotation=30)

    # Cross vs within pie
    axes[1].pie(
        [cross_count, len(audit) - cross_count],
        labels=["Cross-cohort", "Within-cohort"],
        autopct="%1.1f%%",
        colors=["#4C72B0", "#DD8452"],
        startangle=90,
    )
    axes[1].set_title("Cross-cohort vs Within-cohort")

    plt.suptitle("Option D — Annotation Consistency Audit", fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT / "fig_audit_overview.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
if audit_path.exists():
    icf_lookup = dict(zip(icf_df["code"], icf_df["title_de"]))
    rows = []
    for rank, rec in enumerate(audit[:15], 1):
        for m in rec["members"]:
            codes_str = ", ".join(
                f"{c} ({icf_lookup.get(c, '?')[:25]})" for c in m["gold_codes"]
            )
            rows.append({
                "Rank": rank,
                "Severity": rec["disagreement_level"].upper(),
                "Sim": f"{rec['max_similarity']:.3f}",
                "Scope": "cross" if rec["cross_cohort"] else "within",
                "Cohort": m["cohort"],
                "Konzept": m["konzept"][:50],
                "Gold codes": codes_str[:60],
            })

    tbl = pd.DataFrame(rows)  # keep default integer index — Rank is just a column

    def color_severity(val):
        colors_map = {"COMPONENT": "#f8d7da", "MIXED": "#ffe5d0",
                      "CHAPTER": "#fff3cd", "GRANULARITY": "#d4edda",
                      "QUALIFIER": "#d1ecf1"}
        return f"background-color: {colors_map.get(val, '')}"

    display(
        tbl.style
          .map(color_severity, subset=["Severity"])
          .set_caption("Top 15 most suspicious annotation inconsistencies (ranked by severity × similarity)")
    )

---
## 10. LLM Model Comparison (RAG method)

All 8 models use the **same RAG retrieval** (qwen3-embedding:8b index, top-30 codes + top-6 examples),
so accuracy differences reflect purely the LLM's ICF coding ability, not retrieval quality.

| Model | Size | Thinking | Provider |
|-------|------|----------|---------|
| `aya-expanse:32b` | 32B | ✗ | Ollama |
| `deepseek-r1:32b` | 32B | ✓ | Ollama |
| `qwen3.6:35b` | 35B | ✓ | Ollama |
| `llama3.3:70b` | 70B | ✗ | Ollama |
| `gemma4:latest` | ~27B | ✗ | Ollama |
| `gpt-oss:120b` | 120B | ✗ | Ollama |
| `qwen2.5:72b` | 72B | ✗ | Ollama |
| `llama4:latest` | 109B MoE | ✗ | Ollama |

In [ ]:
# ── Load per-model results ──────────────────────────────────────────────────
COMPARISON_MODELS = [
    {"name": "aya-expanse:32b",  "slug": "aya-expanse_32b"},
    {"name": "deepseek-r1:32b",  "slug": "deepseek-r1_32b"},
    {"name": "qwen3.6:35b",      "slug": "qwen3.6_35b"},
    {"name": "llama3.3:70b",     "slug": "llama3.3_70b"},
    {"name": "gemma4:latest",    "slug": "gemma4_latest"},
    {"name": "gpt-oss:120b",     "slug": "gpt-oss_120b"},
    {"name": "qwen2.5:72b",      "slug": "qwen2.5_72b"},
    {"name": "llama4:latest",    "slug": "llama4_latest"},
]

COLORS_8 = sns.color_palette("tab10", 8)
MODEL_PALETTE = {cfg["slug"]: COLORS_8[i] for i, cfg in enumerate(COMPARISON_MODELS)}

model_results = {}
for cfg in COMPARISON_MODELS:
    path = OUTPUT / f"results_rag_{cfg['slug']}.jsonl"
    if not path.exists():
        print(f"  ⚠  {path.name} not found")
        continue
    rows = []
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            scores = score_prediction(rec["predicted_codes"], rec["gold_codes"])
            rec.update(scores)
            rows.append(rec)
    model_results[cfg["slug"]] = rows
    print(f"  Loaded {len(rows):4d} rows  ← {cfg['name']}")

print(f"\n{len(model_results)} / {len(COMPARISON_MODELS)} model result files found.")

In [ ]:
# ── Accuracy table: one row per cohort, one column group per model ──────────
if model_results:
    slug_to_name = {cfg["slug"]: cfg["name"] for cfg in COMPARISON_MODELS}

    frames = []
    for slug, rows in model_results.items():
        s = summary_df(rows)[["Exact %", "One-level-up %", "Component %"]]
        s.columns = pd.MultiIndex.from_tuples(
            [(slug_to_name[slug], col) for col in s.columns]
        )
        frames.append(s)

    model_table = pd.concat(frames, axis=1)
    model_table.to_csv(OUTPUT / "llm_comparison_table.csv")

    display(
        model_table.style
          .format("{:.1f}")
          .background_gradient(cmap="YlGn", axis=None)
          .set_caption("RAG accuracy (%) per model × cohort × metric")
    )
    print("Table saved → output/llm_comparison_table.csv")
else:
    print("No model results loaded — run 08_llm_comparison.py first.")

In [ ]:
# ── Grouped bar chart: Exact Match Accuracy per cohort per model ────────────
if model_results:
    metrics_to_plot = ["Exact %", "One-level-up %", "Component %"]
    cohorts_plot = [c for c in COHORT_ORDER if c in model_table.index]
    n_models = len(model_results)
    bar_w = 0.18

    fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)

    for ax, metric in zip(axes, metrics_to_plot):
        x = np.arange(len(cohorts_plot))
        for i, cfg in enumerate([c for c in COMPARISON_MODELS if c["slug"] in model_results]):
            model_name = cfg["name"]
            slug = cfg["slug"]
            vals = []
            for cohort in cohorts_plot:
                try:
                    v = model_table.loc[cohort, (model_name, metric)]
                except KeyError:
                    v = 0
                vals.append(v)
            offset = (i - (n_models - 1) / 2) * bar_w
            bars = ax.bar(x + offset, vals, bar_w,
                          label=model_name,
                          color=MODEL_PALETTE.get(slug, f"C{i}"),
                          edgecolor="white", linewidth=0.5)
            for bar, v in zip(bars, vals):
                if v > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.7,
                            f"{v:.0f}", ha="center", va="bottom",
                            fontsize=7, color="#333")

        ax.set_title(metric, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(cohorts_plot, rotation=20, ha="right")
        ax.set_ylabel("Accuracy (%)")
        ax.set_ylim(0, 112)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g%%"))
        ax.legend(fontsize=7.5, loc="upper right")

    fig.suptitle("LLM Model Comparison — RAG Linking (same retrieval, different LLMs)",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT / "fig_model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved → output/fig_model_comparison.png")

In [ ]:
# ── ALL-cohort summary: rank models by exact match ─────────────────────────
if model_results:
    slug_to_name = {cfg["slug"]: cfg["name"] for cfg in COMPARISON_MODELS}
    rank_rows = []
    for slug, rows in model_results.items():
        df_m = pd.DataFrame(rows)
        rank_rows.append({
            "Model": slug_to_name[slug],
            "N": len(df_m),
            "Exact %": df_m["exact"].mean() * 100,
            "One-level-up %": df_m["one_level_up"].mean() * 100,
            "Component %": df_m["component"].mean() * 100,
            "Empty preds": (df_m["predicted_codes"].apply(len) == 0).sum(),
        })

    rank_df = (
        pd.DataFrame(rank_rows)
          .set_index("Model")
          .sort_values("Exact %", ascending=False)
    )

    display(
        rank_df.style
          .format("{:.1f}", subset=["Exact %", "One-level-up %", "Component %"])
          .format("{:.0f}", subset=["N", "Empty preds"])
          .background_gradient(cmap="YlGn", subset=["Exact %", "One-level-up %", "Component %"])
          .set_caption("ALL-cohort summary — 8 models ranked by exact match accuracy")
    )

    best_model = rank_df["Exact %"].idxmax()
    print(f"\n🏆 Best model (exact match): {best_model}  →  {rank_df.loc[best_model, 'Exact %']:.1f}%")
    print(f"   Best 1-up:      {rank_df['One-level-up %'].max():.1f}%  ({rank_df['One-level-up %'].idxmax()})")
    print(f"   Best component: {rank_df['Component %'].max():.1f}%  ({rank_df['Component %'].idxmax()})")

---
## 11. Embedding Model Comparison

**Question:** Does the embedding model affect retrieval quality and downstream accuracy?

Same LLM (`qwen3.6:35b`) and same prompt structure. Only the embedding model changes:
- **Baseline:** `qwen3-embedding:8b` (4,096-dim) — current pipeline
- **Challenger:** `bge-m3:latest` (1,024-dim) — strong multilingual alternative

In [ ]:
# ── Load embedding comparison results ──────────────────────────────────────
embed_baseline_path   = OUTPUT / "results_option_c.jsonl"
embed_challenger_path = OUTPUT / "results_option_c_bge-m3_latest.jsonl"

def load_embed_results(path):
    if not path.exists():
        return None
    rows = []
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            scores = score_prediction(rec["predicted_codes"], rec["gold_codes"])
            rec.update(scores)
            rows.append(rec)
    return rows

embed_baseline   = load_embed_results(embed_baseline_path)
embed_challenger = load_embed_results(embed_challenger_path)

if embed_baseline and embed_challenger:
    labels = ["qwen3-embedding:8b\n(4096-dim)", "bge-m3:latest\n(1024-dim)"]
    results_list = [embed_baseline, embed_challenger]
    colors = ["#4C72B0", "#DD8452"]

    metrics = ["Exact %", "One-level-up %", "Component %"]
    metric_keys = ["exact", "one_level_up", "component"]

    # Summary table
    rows_out = []
    for label, res in zip(["qwen3-embedding:8b (4096-dim)", "bge-m3:latest (1024-dim)"], results_list):
        df_e = pd.DataFrame(res)
        rows_out.append({
            "Embedding model": label,
            "N": len(df_e),
            "Exact %": df_e["exact"].mean() * 100,
            "One-level-up %": df_e["one_level_up"].mean() * 100,
            "Component %": df_e["component"].mean() * 100,
        })

    embed_df = pd.DataFrame(rows_out).set_index("Embedding model")
    delta_row = embed_df.iloc[1] - embed_df.iloc[0]
    embed_df.loc["Δ (bge-m3 − qwen3-embedding)"] = delta_row

    display(
        embed_df.style
          .format("{:.1f}", subset=["Exact %", "One-level-up %", "Component %"])
          .format("{:.0f}", subset=["N"])
          .background_gradient(cmap="RdYlGn", subset=["Exact %", "One-level-up %", "Component %"], axis=None)
          .set_caption("Embedding model comparison — same LLM (qwen3.6:35b), same 470 test rows")
    )

    # Bar chart
    x = np.arange(len(metrics))
    bar_w = 0.35
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, (label, res, color) in enumerate(zip(labels, results_list, colors)):
        df_e = pd.DataFrame(res)
        vals = [df_e[k].mean() * 100 for k in metric_keys]
        offset = (i - 0.5) * bar_w
        bars = ax.bar(x + offset, vals, bar_w, label=label, color=color, edgecolor="white")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f"{v:.1f}%", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.set_title("Embedding Model Comparison (RAG, qwen3.6:35b LLM)", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / "fig_embedding_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved → output/fig_embedding_comparison.png")

    winner = "qwen3-embedding:8b" if embed_df.loc["Δ (bge-m3 − qwen3-embedding)", "Exact %"] < 0 else "bge-m3"
    delta_exact = embed_df.loc["Δ (bge-m3 − qwen3-embedding)", "Exact %"]
    print(f"\n→ {winner} wins on exact match by {abs(delta_exact):.1f} pp")
else:
    print("⚠ Run 09_embedding_comparison.py first to generate embedding comparison results.")

---
## 12. Summary

In [ ]:
print("=" * 65)
print("RESULTS SUMMARY")
print("=" * 65)

# Method comparison
for label, res in available.items():
    all_rows = pd.DataFrame(res)
    print(f"\n{label}")
    print(f"  Exact match:     {all_rows['exact'].mean()*100:.1f}%")
    print(f"  One-level-up:    {all_rows['one_level_up'].mean()*100:.1f}%")
    print(f"  Component match: {all_rows['component'].mean()*100:.1f}%")
    print(f"  N test rows:     {len(all_rows)}")

# Option D
if audit_path.exists():
    print(f"\nOption D — Consistency audit:")
    print(f"  Inconsistent groups: {len(audit)}")
    print(f"  Cross-cohort:        {cross_count}")
    print(f"  Within-cohort:       {len(audit) - cross_count}")

# LLM comparison (all 8 models)
if model_results:
    slug_to_name = {cfg["slug"]: cfg["name"] for cfg in COMPARISON_MODELS}
    print(f"\nLLM Model Comparison (RAG) — ranked by exact match:")
    model_summary = []
    for slug, rows in model_results.items():
        df_m = pd.DataFrame(rows)
        model_summary.append((
            slug_to_name[slug],
            df_m["exact"].mean()*100,
            df_m["one_level_up"].mean()*100,
            df_m["component"].mean()*100,
        ))
    for name, e, u, c in sorted(model_summary, key=lambda x: -x[1]):
        print(f"  {name:<22}  exact={e:.1f}%  1-up={u:.1f}%  comp={c:.1f}%")

# Embedding comparison
if embed_baseline and embed_challenger:
    b = pd.DataFrame(embed_baseline)
    ch = pd.DataFrame(embed_challenger)
    print(f"\nEmbedding Comparison (RAG, qwen3.6:35b LLM):")
    print(f"  qwen3-embedding:8b (4096-dim):  exact={b['exact'].mean()*100:.1f}%  comp={b['component'].mean()*100:.1f}%")
    print(f"  bge-m3:latest (1024-dim):       exact={ch['exact'].mean()*100:.1f}%  comp={ch['component'].mean()*100:.1f}%")
    print(f"  → qwen3-embedding:8b wins by {(b['exact'].mean()-ch['exact'].mean())*100:.1f} pp on exact match")

print("\n" + "=" * 65)